# WeatherMesh-3 inference

## Environment

In [10]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt


def find_repo():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "huge").is_dir() and (base / "constants").is_dir():
            return base
    return Path.cwd()


REPO = find_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

DATA = REPO / "huge" / "proc" / "haoxing_data" / "wm3" / "data"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("repo:", REPO)
print("torch:", torch.__version__, "| device:", device)

repo: /Users/sudhanva/Desktop/WindBorne/WeatherMesh-3
torch: 2.5.1 | device: cpu


## Input and output meshes
Same construction as `model.get_WeatherMesh3`. Both inputs resolve to 28 (`levels_medium`) internal levels and `n_vars = 157`.

In [11]:
from meshes import LatLonGrid
from utils import levels_gfs, levels_hres, levels_medium, interp_levels, get_date

extra_in = ["45_tcc", "168_2d", "246_100u", "247_100v"]
extra_out = ["15_msnswrf", "45_tcc", "168_2d", "246_100u", "247_100v",
             "142_lsp", "143_cp", "201_mx2t", "202_mn2t",
             "142_lsp-6h", "143_cp-6h", "201_mx2t-6h", "202_mn2t-6h"]

gfs_mesh = LatLonGrid(source="neogfs-25", extra_sfc_vars=extra_in,
                      extra_sfc_pad=len(extra_out) - len(extra_in),
                      input_levels=levels_gfs, levels=levels_medium)
hres_mesh = LatLonGrid(source="neohres-20", extra_sfc_vars=extra_in,
                       extra_sfc_pad=len(extra_out) - len(extra_in),
                       input_levels=levels_hres, levels=levels_medium)
era_mesh = LatLonGrid(source="era5-28", extra_sfc_vars=extra_out, levels=levels_medium)

for name, m in [("neogfs", gfs_mesh), ("neohres", hres_mesh), ("era5-out", era_mesh)]:
    print(f"{name:9} input_levels={len(m.input_levels):2} internal_levels={m.n_levels} "
          f"n_pr={m.n_pr} n_sfc={m.n_sfc} n_vars={m.n_vars} shape={tuple(m.shape())}")

Set string_id for LatLonGrid to neogfs-25>28p5s8z9r0.25-VfOP
Set string_id for LatLonGrid to neohres-20>28p5s8z9r0.25-VfOP
Set string_id for LatLonGrid to era5-28>28p5s17z0r0.25-kUAg
neogfs    input_levels=25 internal_levels=28 n_pr=140 n_sfc=17 n_vars=157 shape=(720, 1440, 157)
neohres   input_levels=20 internal_levels=28 n_pr=140 n_sfc=17 n_vars=157 shape=(720, 1440, 157)
era5-out  input_levels=28 internal_levels=28 n_pr=140 n_sfc=17 n_vars=157 shape=(720, 1440, 157)


## Load the raw sample

In [12]:
def load_source(source):
    p = next((DATA / source / "f000").glob("**/*.npz"))
    z = np.load(p)
    extras = [np.load(next((DATA / source / "extra" / v).glob("**/*.npz")))["x"]
              for v in extra_in]
    return z["pr"], z["sfc"], np.stack(extras, axis=-1), int(p.stem)


pr_g, sfc_g, ex_g, TS = load_source("neogfs")
pr_h, sfc_h, ex_h, _ = load_source("neohres")

print("neogfs  pr", pr_g.shape, "sfc", sfc_g.shape, "extra", ex_g.shape)
print("neohres pr", pr_h.shape, "sfc", sfc_h.shape, "extra", ex_h.shape)
print("timestamp", TS, "->", get_date(TS).strftime("%Y-%m-%d %H:%MZ"))

neogfs  pr (721, 1440, 5, 25) sfc (721, 1440, 4) extra (721, 1440, 4)
neohres pr (721, 1440, 5, 20) sfc (721, 1440, 4) extra (721, 1440, 4)
timestamp 1741305600 -> 2025-03-07 00:00Z


## Reconstruct the encoder inputs
Trim the south pole (721 -> 720), lay pressure out variable-major / level-minor, append `[core sfc | extra sfc | zeropad]`, then lift the native levels (25 / 20) to the 28 internal levels with `interp_levels`.

In [13]:
def build_input(mesh, pr, sfc, ex):
    pr = pr[:720].astype(np.float32)
    sfc = sfc[:720].astype(np.float32)
    ex = ex[:720].astype(np.float32)
    H, W, nv, nl = pr.shape
    pr_flat = pr.reshape(H, W, nv * nl)
    pad = np.zeros((H, W, mesh.extra_sfc_pad), np.float32)
    sfc_block = np.concatenate([sfc, ex, pad], axis=-1)
    x_in = torch.from_numpy(np.concatenate([pr_flat, sfc_block], axis=-1))
    return interp_levels(x_in, mesh, mesh.input_levels, mesh.levels)


gx = build_input(gfs_mesh, pr_g, sfc_g, ex_g)
hx = build_input(hres_mesh, pr_h, sfc_h, ex_h)
t0s = torch.tensor([TS])

zpad = gx[..., gfs_mesh.n_pr + len(gfs_mesh.core_sfc_vars) + len(extra_in):]
print("gfs  input tensor:", tuple(gx.shape))
print("hres input tensor:", tuple(hx.shape))
print("zeropad channels :", tuple(zpad.shape), "all zero:", bool(torch.all(zpad == 0)))

gfs  input tensor: (720, 1440, 157)
hres input tensor: (720, 1440, 157)
zeropad channels : (720, 1440, 9) all zero: True


## Verify the reconstruction
De-normalize with each mesh's own `normalization_matrix_mean/std` and confirm the fields are physical.

In [14]:
def denorm_channel(x, mesh, idx):
    return x[..., idx].numpy() * mesh.normalization_matrix_std[idx] + mesh.normalization_matrix_mean[idx]


i2t = gfs_mesh.n_pr + gfs_mesh.core_sfc_vars.index("167_2t")
iz500 = gfs_mesh.pressure_vars.index("129_z") * gfs_mesh.n_levels + gfs_mesh.levels.index(500)
t2m = denorm_channel(gx, gfs_mesh, i2t)
z500 = denorm_channel(gx, gfs_mesh, iz500) / 9.80665

print(f"2m temperature : {t2m.min()-273.15:6.1f} .. {t2m.max()-273.15:6.1f} degC (mean {t2m.mean()-273.15:.2f})")
print(f"500 hPa height : {z500.min():6.0f} .. {z500.max():6.0f} m")
print(f"normed pr block: mean {gx[..., :gfs_mesh.n_pr].mean():+.3f}  std {gx[..., :gfs_mesh.n_pr].std():.3f}")

2m temperature :  -60.9 ..   37.5 degC (mean 3.44)
500 hPa height :   4774 ..   5982 m
normed pr block: mean -0.346  std 1.374


## Load WeatherMesh-3

In [15]:
from model import get_WeatherMesh3

model = get_WeatherMesh3("model/WeatherMesh3.pt").to(device).eval()
print("parameters:", f"{sum(p.numel() for p in model.parameters())/1e6:.1f} M")

ModuleNotFoundError: No module named 'matepoint'

## Run the forecast
`todo=[6]` expands to `E,P6,D` (one 6-hour step). Encoders run in parallel and blend `0.1*gfs + 0.9*hres`.

In [ ]:
x = [gx[None].to(device), hx[None].to(device), t0s.to(device)]
with torch.no_grad():
    out = model(x, [6])

pred = out[6][0]
print("prediction:", tuple(pred.shape), "| latent_l2:", float(out["latent_l2"]))
print("valid time:", get_date(TS + 6 * 3600).strftime("%Y-%m-%d %H:%MZ"))

## Decode the ERA5 output
Absolute predicted state (not a delta), laid out as `era_mesh.full_varlist` including the 13 extra output variables.

In [ ]:
mean = era_mesh.normalization_matrix_mean
std = era_mesh.normalization_matrix_std
real = pred[0].float().cpu().numpy() * std + mean


def field(name):
    return real[..., era_mesh.full_varlist.index(name)]


print("surface output vars:", era_mesh.sfc_vars)
print(f"forecast 2m temp : {field('167_2t').min()-273.15:.1f} .. {field('167_2t').max()-273.15:.1f} degC")
print(f"forecast mslp    : {field('151_msl').min()/100:.0f} .. {field('151_msl').max()/100:.0f} hPa")

## Visualize the forecast
Predicted fields plus the 6-hour tendency (forecast minus the t0 analysis).

In [ ]:
EXT = [0, 360, -90, 90]


def show_map(f, title, cmap="viridis", ax=None):
    ax = ax or plt.gca()
    im = ax.imshow(np.asarray(f, np.float32), extent=EXT, origin="upper", aspect="auto", cmap=cmap)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.85)


in2t = denorm_channel(gx, gfs_mesh, i2t)[:720] - 273.15
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
show_map(field("167_2t") - 273.15, "forecast +6h 2m temp (degC)", "RdBu_r", axes[0, 0])
show_map(field("151_msl") / 100, "forecast +6h mslp (hPa)", "viridis", axes[0, 1])
show_map(field("142_lsp"), "forecast +6h large-scale precip", "Blues", axes[1, 0])
show_map((field("167_2t") - 273.15) - in2t, "6h 2m-temp tendency (degC)", "coolwarm", axes[1, 1])
fig.tight_layout()
plt.show()

## Notes & caveats
- **Reconstructed input.** The repo withholds its data-processing; the mesh build, input reconstruction and de-norm checks above validate the shapes/ordering independently.
- **No verification target.** This sample has only the `t0` analysis inputs — no ERA5 truth at `t0+6h` — so only plausibility and the forecast-vs-analysis tendency, not RMSE.
- **Interpolation is done in normalized space**, as in training.
- **The `extra/` files are byte-identical across `neogfs` and `neohres`** in this sample.
- **Memory.** Full 720x1440 at `latent_size=1024` is large; use `.half()` or `checkpoint_type='matepoint'` if you hit OOM.